Generate a random list of nodes and edges, specifying the strengths distributions

In [1]:
# auto-reload the packages at every run
%load_ext autoreload
%autoreload 2

#display all the results not only the last one
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [4]:
N = 10 #int(2.9 * 1e5)
# E = int(2.3 * 1e6)

In [5]:
import numpy as np
import numpy.random as npr
# sample the strengths in from the log-normal distribution
mu_in, scale_in = 9.145050968796784, 3.0501786627011396
npr.seed(0)
logn_sample = lambda mu, scale: npr.lognormal(mean=mu, sigma=scale, size=N)
prop_in = logn_sample(mu_in, scale_in)

# sample the strengths out from the log-normal distribution
mu_out, scale_out = 9.753643239750687, 3.2045983690423174
stre_out_method = 'correlated'
if stre_out_method == 'uncorrelated':
    prop_out = logn_sample(mu_out, scale_out)
elif stre_out_method == 'correlated':
    mu_eps, scale_eps = mu_out - mu_in, np.sqrt(scale_out**2 - scale_in**2)
    eps = logn_sample(mu_eps, scale_eps)
    prop_out = prop_in * eps

In [6]:
# # plot the distributions of the prop_in and prop_out
# import matplotlib.pyplot as plt
# plt.figure(figsize=(10, 5))
# plt.hist(np.log(prop_in), bins=100, alpha=0.5, label='prop_in', density=True)
# plt.hist(np.log(prop_out), bins=100, alpha=0.5, label='prop_out', density=True)
# plt.xlabel('Strength')
# plt.ylabel('Density')
# plt.legend()
# plt.title('Distribution of Strengths')

In [7]:
from graph_ensembles import sparse as ge

In [20]:
#v = np.arange(len(prop_out), dtype=np.int64)
param = 2e-13 #1.8996372e-17 #1.8996372e-17
kwargs_model = {
                "num_vertices" : N,
                "prop_out" : prop_out,
                "prop_in" : prop_in,
                "param" : param,
                "selfloops" : False,
                "name" : 'Invariant',
                "level" : 0,
                "seed" : 0,
                "perc_intra_nodes" : 1,
                "fit_method" : 'num_edges_intra',
                "corpkey" : False,
                "test_graph" : "intra",
                } #git hub repositories
model = ge.ScaleInvariantModel(**kwargs_model)

In [21]:
self = model
# self.num_edges_fit_fun([1.9e-17])
# exp_edges_f_jac(p_jac_ij, param, prop_out, prop_in, prop_dyad, selfloops)

Parallelized Code

In [29]:
from numba import njit, prange, get_num_threads, get_thread_id
import numpy as np
from numba.typed import List

@njit(parallel=False)
def block_parallel_sample(p_ij, param, prop_out, prop_in, prop_dyad, selfloops):
    N = len(prop_out)
    total_ops = N * N
    num_blocks = get_num_threads()
    block_size = total_ops // num_blocks

    # Preallocate buffers for each block/thread
    thread_rows = List()
    thread_cols = List()

    for _ in range(num_blocks):
        thread_rows.append(List.empty_list(np.int64))
        thread_cols.append(List.empty_list(np.int64))

    np.random.seed(1)
    for block in prange(num_blocks):
        start = block * block_size
        end = (block + 1) * block_size if block < num_blocks - 1 else total_ops
        thread_id = get_thread_id()
        for flat_idx in range(start, end):
            i = flat_idx // N
            j = flat_idx % N
            if not selfloops and i == j:
                continue
            p = p_ij(param, prop_out[i], prop_in[j], prop_dyad(i, j))
            if np.random.random() < p:
                thread_rows[thread_id].append(i)
                thread_cols[thread_id].append(j)

    return thread_rows, thread_cols

In [30]:
rows, cols = block_parallel_sample(
    model.p_ij, 
    model.param, 
    model.prop_out, 
    model.prop_in,
    model.prop_dyad,
    model.selfloops
)
from itertools import chain
rows = np.fromiter(chain.from_iterable(rows), dtype=np.int64)
cols = np.fromiter(chain.from_iterable(cols), dtype=np.int64)

In [31]:
rows
cols

array([0, 0, 0, 2, 3, 3, 4, 4, 4, 6])

array([3, 4, 6, 0, 0, 4, 0, 2, 3, 3])

This is the code without parallelization copied from the ensembles

In [26]:
from numba import njit, prange
from numba.typed import List
@njit()  # pragma: no cover
def _binary_sample(p_ij, param, prop_out, prop_in, prop_dyad, selfloops):
    """Sample from the ensemble."""
    rows = List()
    cols = List()

    N = len(prop_out)
    np.random.seed(1)
    for i in prange(N):
        p_out_i = prop_out[i]
        for j in range(N):
            if (i != j):
                p_in_j = prop_in[j]
                p = p_ij(param, p_out_i, p_in_j, prop_dyad(i, j))
                if np.random.random() < p:
                    rows.append(i)
                    cols.append(j)

        # if i % 10 == 0: print(f'-i: {i}',)
        
    return rows, cols

@njit()  # pragma: no cover
def leo_binary_sample(p_ij, param, prop_out, prop_in, prop_dyad, selfloops):
    """Sample from the ensemble."""
    rows = []
    cols = []
    np.random.seed(1)
    for i, p_out_i in enumerate(prop_out):
        for j, p_in_j in enumerate(prop_in):
            if (i != j) | selfloops:
                p = p_ij(param, p_out_i, p_in_j, prop_dyad(i, j))
                if np.random.random() < p:
                    rows.append(i)
                    cols.append(j)

    return rows, cols

In [27]:
rows_fail, cols_fail = leo_binary_sample(
    self.p_ij,
    self.param,
    self.prop_out,
    self.prop_in,
    self.prop_dyad,
    self.selfloops,
)
# ipython_pygments_lexers

In [ ]:
rows
rows_fail

cols
cols_fail

array([0, 0, 0, 2, 3, 3, 4, 4, 4, 6])

[0, 0, 0, 2, 3, 3, 4, 4, 4, 6]

array([3, 4, 6, 0, 0, 4, 0, 2, 3, 3])

[3, 4, 6, 0, 0, 4, 0, 2, 3, 3]

240

In [ ]:
from graph_ensembles.utils import parallel_sample

In [ ]:
import os
rows, cols = parallel_sample(
    self.p_ij,
    self.param,
    self.prop_out,
    self.prop_in,
    self.prop_dyad,
    self.selfloops,
    num_procs = os.cpu_count() - 1
)
# rows, cols

In [11]:
rows_fail[:10]
cols_fail[:10]

ListType[int64]([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

ListType[int64]([3, 1240, 3118, 3230, 4971, 7580, 9863, 11684, 16362, 17992])

In [ ]:
idx = np.lexsort((cols_fail, rows_fail))
rows_fail = np.array(rows_fail)[idx]
cols_fail = np.array(cols_fail)[idx]

idx = np.lexsort((cols_fast, rows_fast))

rows_fast = rows_fast[idx]
cols_fast = cols_fast[idx]

if np.all(rows_fast == rows_fail) and np.all(cols_fast == cols_fail):
    print(f'-rows and cols: {True}',)
else:
    if np.all(rows_fast == rows_fail):
        print(f'-rows: {True}',)
    else:
        rows_fast == rows_fail

    if np.all(cols_fast == cols_fail):
        print(f'-cols: {True}',)
    else:
        cols_fast == cols_fail

    rows_fail
    rows_fast

    cols_fail
    cols_fast

In [ ]:
import numpy as np
from concurrent.futures import ThreadPoolExecutor

def worker(start, end, N, p_ij, param, prop_out, prop_in, prop_dyad, selfloops):
    rows = []
    cols = []
    worker_id = current_thread().name
    print(f'Worker {worker_id} started: processing indices {start} to {end}')
    for flat_idx in range(start, end):
        # Convert flat index to 2D indices, via CSR indexing
        i = flat_idx // N
        j = flat_idx % N

        # if selfloops is False and i == j --> skip this pair
        if not selfloops and i == j:
            continue
        p = p_ij(param, prop_out[i], prop_in[j], prop_dyad(i, j))
        if np.random.random() < p:
            rows.append(i)
            cols.append(j)
        # Optionally, print progress for each worker
        # print(f'Worker {worker_id}, rows, cols: {rows}, {cols}')
    print(f'Worker {worker_id} finished.')
    return np.array(rows, dtype=np.int64), np.array(cols, dtype=np.int64)

def parallel_sample(p_ij, param, prop_out, prop_in, prop_dyad, selfloops, num_threads=3):
    N = len(prop_out)
    total_ops = N * N  # or N * (N - 1) if not selfloops
    num_chunks = total_ops // num_threads

    futures = []
    with ThreadPoolExecutor(max_workers=num_threads, thread_name_prefix='agent') as executor:
        for t in range(num_threads):
            start = t * num_chunks
            end = (t + 1) * num_chunks if t < num_threads - 1 else total_ops
            futures.append(executor.submit(
                worker, start, end, N, p_ij, param, prop_out, prop_in, prop_dyad, selfloops
            ))

    rows = []
    cols = []
    for f in futures:
        r, c = f.result()
        rows.append(r)
        cols.append(c)
    return np.concatenate(rows), np.concatenate(cols)

In [ ]:
import numpy as np
import numba
from numba import njit, prange, config

@njit(parallel=True)
def _fixed_binary_sample(p_ij, param, prop_out, prop_in, prop_dyad, selfloops):
    N = len(prop_out)
    
    # Pre-allocate thread-local buffers as arrays
    n_threads = config.NUMBA_DEFAULT_NUM_THREADS
    max_edges_per_thread = 2000  # Adjust based on expected density
    thread_buffers_rows = np.empty((n_threads, max_edges_per_thread), dtype=np.int64)
    thread_buffers_cols = np.empty((n_threads, max_edges_per_thread), dtype=np.int64)
    
    # counts how many edges were sampled by each thread
    counts = np.zeros(n_threads, dtype=np.int64)

    np.random.seed(0)  # Set seed for reproducibility when parallel = False
    # Parallel sampling
    for i in prange(N):
        thread_id = numba.get_thread_id()
        # print(f'-thread_id: {thread_id}',)
        p_out_i = prop_out[i]
        
        for j in range(N):
            if i != j:
                p_in_j = prop_in[j]
                p = p_ij(param, p_out_i, p_in_j, prop_dyad(i, j))
                if np.random.random() < p:
                    if counts[thread_id] < max_edges_per_thread:
                        thread_buffers_rows[thread_id, counts[thread_id]] = i
                        thread_buffers_cols[thread_id, counts[thread_id]] = j
                        counts[thread_id] += 1

    # Concatenate results
    total_edges = np.sum(counts)
    rows = np.empty(total_edges, dtype=np.int64)
    cols = np.empty(total_edges, dtype=np.int64)
    
    idx = 0
    for t in range(n_threads):
        n = counts[t]
        rows[idx:idx+n] = thread_buffers_rows[t, :n]
        cols[idx:idx+n] = thread_buffers_cols[t, :n]
        idx += n

    return rows, cols


In [ ]:
from numba import njit, prange

@njit(parallel=True)
def _fast_binary_sample(p_ij, param, prop_out, prop_in, prop_dyad, selfloops, randmat):
    N = len(prop_out)
    edge_counts = np.zeros(N, dtype=np.int64)
    
    # First pass: count edges per i
    for i in prange(N):
        for j in range(N):
            if i != j:
                p = p_ij(param, prop_out[i], prop_in[j], prop_dyad(i, j))
                if randmat[i,j] < p:
                    edge_counts[i] += 1
    total_edges = np.sum(edge_counts)
    
    # Compute prefix sum for unique indexing
    idxs = np.zeros(N, dtype=np.int64)
    idxs[1:] = np.cumsum(edge_counts)[:-1]
    rows = np.empty(total_edges, dtype=np.int64)
    cols = np.empty(total_edges, dtype=np.int64)
    
    # Second pass: fill arrays
    for i in prange(N):
        idx = idxs[i]
        for j in range(N):
            if i != j:
                p = p_ij(param, prop_out[i], prop_in[j], prop_dyad(i, j))
                if randmat[i,j] < p:
                    rows[idx] = i
                    cols[idx] = j
                    idx += 1
    return rows, cols

In [ ]:
rng = np.random.default_rng(0)
randmat = rng.random((N, N))

rows_fast, cols_fast = _fast_binary_sample(
    self.p_ij,
    self.param,
    self.prop_out,
    self.prop_in,
    self.prop_dyad,
    self.selfloops,
    randmat
)
# ipython_pygments_lexers